In [1]:
%matplotlib widget
import inspect
import re
def debugPrint(x):
    frame = inspect.currentframe().f_back
    s = inspect.getframeinfo(frame).code_context[0]
    r = re.search(r"\((.*)\)", s).group(1)
    print("{} [{}] = {}".format(r,type(x).__name__, x))
       
import torch
import numpy as np
import warp as wp

# Initialize Warp
wp.config.verify_autograd_array_access = False
wp.config.verbose = False
wp.init()

from sphWarpCore import radiusSearchCompactHashMap, sphOperation_warp
from sphWarpCore.enumTypes import *

import matplotlib.pyplot as plt
from demo_util import *
from warpPlot import *

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
device = torch.device('cpu')
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
targetNumNeighbors = 50
nx = 512
dim = 2
numParticles = nx**dim

warpOnly = False
periodic = True

kernel = KernelFunctions.Wendland2
supportMode = SupportScheme.Gather

markerSize = 8 if warpOnly else 2
gridVisualization = False 
gridResolution = 128
dx = 2.0 / nx

In [3]:
particleState, domain, adjacency, neighborhood, simulationState, measurements = prepData(nx, targetNumNeighbors, dim, device, periodic, warpOnly)
apparentVolume, crkDensity, crkState = computeCRKFactors(particleState, domain, kernel, adjacency = adjacency)

Module sphWarpCore.radiusSearch.wp_compactHash e2c9126 load on device 'cuda:0' took 4.43 ms  (cached)
Module sphWarpCore.operations.wp_density 6e0e3a3 load on device 'cuda:0' took 2.38 ms  (cached)
Module sphWarpCore.crk.crk_volume 0f6d8bf load on device 'cuda:0' took 1.28 ms  (cached)
Module sphWarpCore.crk.crk_moments 9d838a2 load on device 'cuda:0' took 1.81 ms  (cached)
Module sphWarpCore.crk.crk_density 8012a37 load on device 'cuda:0' took 2.32 ms  (cached)


In [4]:
f_linear = particleState.positions[:,0] * 5 + 10
f_grad_x = torch.full_like(f_linear, 5.0)
f_grad_y = torch.zeros_like(f_linear)

linear_gradient_warp = warpOperation(
    queryParticles = particleState,
    queryValues = f_linear,
    operationProperties=OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Gradient,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = adjacency,
    domain = domain,
)

print("Linear Gradient (WarpSPH): ", linear_gradient_warp)

mean_error_x = torch.mean(torch.abs(linear_gradient_warp[:,0] - f_grad_x))
mean_error_y = torch.mean(torch.abs(linear_gradient_warp[:,1] - f_grad_y))

print("Mean Absolute Error in X Gradient: ", mean_error_x.item())
print("Mean Absolute Error in Y Gradient: ", mean_error_y.item())

Module sphWarpCore.operations.wp_gradient 588bd4a load on device 'cuda:0' took 3.05 ms  (cached)
Linear Gradient (WarpSPH):  tensor([[-870.9697,   27.1571],
        [-302.8944,   19.5059],
        [ -44.0832,    4.3140],
        ...,
        [ -79.8694,    5.2375],
        [-354.2237,   20.7125],
        [-853.1558,   44.1675]], device='cuda:0')
Mean Absolute Error in X Gradient:  5.047910690307617
Mean Absolute Error in Y Gradient:  0.19542640447616577


In [5]:
print("Testing Linear Interpolation with WarpSPH...")
print("Input Function: f(x,y) = 5x + 10", 'Min: ', torch.min(f_linear), "Max: ", torch.max(f_linear))
linear_interp = warpOperation(
    queryParticles = particleState,
    queryValues = f_linear,
    operationProperties=OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Interpolate,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = None,
    domain = domain,
)
print("Linear Interpolation (WarpSPH): ", linear_interp, "Min: ", torch.min(linear_interp), "Max: ", torch.max(linear_interp))

Testing Linear Interpolation with WarpSPH...
Input Function: f(x,y) = 5x + 10 Min:  tensor(5.0027, device='cuda:0') Max:  tensor(14.9968, device='cuda:0')
Module sphWarpCore.operations_grid.wp_interpolate_grid 2d6d80a load on device 'cuda:0' took 4.25 ms  (cached)
Linear Interpolation (WarpSPH):  tensor([ 8.1326,  5.6960,  5.1579,  ..., 15.0429, 14.5495, 12.0443],
       device='cuda:0') Min:  tensor(4.8551, device='cuda:0') Max:  tensor(15.6038, device='cuda:0')


In [6]:
f = torch.randn(numParticles, device=device, dtype=torch.float32)

f_smoothed = f.clone()

for _ in range(4):
    f_smoothed = warpOperation(
        queryParticles = particleState,
        queryValues = f_smoothed,
        operationProperties = OperationProperties(
            kernel = kernel,
            supportMode = supportMode,
            operation = WarpOperation.Interpolate,
        ),
        adjacency = None,
        domain = domain,
    )

gradient_warp = warpOperation(
    queryParticles = particleState,
    queryValues = f_smoothed,
    operationProperties = OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Gradient,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = adjacency,
    domain = domain,
)


if not warpOnly:
    gradient_diffSPH = SPHOperation(
        simulationState,
        quantity = f_smoothed,
        kernel = KernelType.Wendland2,
        neighborhood = neighborhood[0],
        kernelValues = neighborhood[1],
        operation=Operation.Gradient,
        gradientMode=GradientMode.Difference,
        supportScheme = SupportScheme.Gather,
        correctionTerms= [],
        positiveDivergence=False
    )



In [7]:
from warpPlot import *

In [8]:
plotter = visualize(
    particleState = particleState,
    domain = domain,
    quantities = {
        "A": f_smoothed,
        "B": -f_smoothed,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = ColorMap.rocket,
            markerSize = 16,
            midPoint = 'median',
            plotTitle = "Visualization of Smoothed Quantity via Grid",
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        ),
        "B": PlottingOptions(
            colorMap = ColorMap.viridis,
            markerSize = 0.1,
            midPoint = 'median',
            plotTitle = "Visualization of Negative Smoothed Quantity via Grid",
            gridVisualization = GridVisualization(
                resolution = 32,
            ),
        ),
    },
    figTitle = "Initial Visualization of Smoothed Quantities",
    mosaic = 'AB',
    figsize= (11,5),
    backend='vispy',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

RFBOutputContext()

In [9]:
import copy
from tqdm.autonotebook import tqdm

tempState = copy.deepcopy(particleState)
quantity = f_smoothed.clone()
try:
    for i in tqdm(range(16)):
        offset = torch.zeros_like(tempState.positions)
        offset[:,0] = 0.5
        tempState.positions += offset
        quantity += 0.02 * torch.randn_like(quantity)
        plotter.updateQuantities(
            {
                "A": quantity,
                "B": -quantity,
            },
            newParticleState=tempState,
            newOptions={
                "A": {
                    "colorMap": ColorMap.Spectral,
                }
            },
            # redraw=True,
            # redrawEvery=2,
            # Keep notebook widgets responsive from sync code.
            # yieldNotebookEvents=True,
            # yieldSeconds=0.02,
        )
        time.sleep(0.1)

    plotter.show()
except KeyboardInterrupt:
    print("Update loop interrupted.")
    # plotter.show()

  0%|          | 0/16 [00:00<?, ?it/s]